In [1]:
import torch
from torch import nn
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [ ]:
#=nb

In [2]:
train = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)
test = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [3]:
train_dataloader = DataLoader(train.data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test.data, batch_size=64, shuffle=True)

In [8]:
len(train_dataloader)

938

In [4]:
device  = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [5]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(100, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 28*28),
            nn.Sigmoid(),
        )
    def forward(self, X):
       return self.model(X) 
g_model = Generator().to(device)
g_model

Generator(
  (model): Sequential(
    (0): Linear(in_features=100, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1024, bias=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=784, bias=True)
    (7): Sigmoid()
  )
)

In [6]:
class Descriminator(nn.Module):
    def __init__(self):
        super(Descriminator, self).__init__()
        self.flatten = nn.Flatten()
        self.model = nn.Sequential(
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512,1024),
            nn.ReLU(),
            nn.Linear(1024,2),
            nn.Sigmoid()
        )
    def forward(self,X):
        X = self.flatten(X)
        return self.model(X)
d_model = Descriminator().to(device)
d_model

Descriminator(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (model): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1024, bias=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=2, bias=True)
    (7): Sigmoid()
  )
)

In [ ]:
loss = nn.BCELoss()
optimizer_g = torch.optim.Adam(g_model.parameters(), lr=0.0002)
optimizer_d = torch.optim.Adam(d_model.parameters(), lr=0.0002)


In [17]:
def train(dataloader,g_model,d_model,loss, optim_d, optim_g,k):
    dec_loss = 0
    gen_loss = 0
    size = len(dataloader)
    g_model.train()
    d_model.train()
    for X in dataloader:
        for i in range(k):
            optim_d.zero_grad()
            noise = torch.randn(X.shape[0],100).to(device)
            real = X.to(device).float()
            gen = g_model(noise)
            dec_real = d_model(real).float()
            dec_gen = d_model(gen).float()
            real_loss = loss(dec_real, torch.ones_like(dec_real))
            gen_loss = loss(dec_gen, torch.zeros_like(dec_gen))
            d_loss = real_loss + gen_loss
            dec_loss += d_loss.item()
            d_loss.backward()
            optim_d.step()
        optim_g.zero_grad()
        noise = torch.randn(X.shape[0],100).to(device)
        gen = g_model(noise)
        dec = d_model(gen)
        g_loss = loss(dec, torch.zeros_like(dec))
        gen_loss += g_loss.item()
        g_loss.backward()
        optim_g.step()
    print(f'Descriminator Loss : {dec_loss/size}, Generator Loss : {gen_loss/size}')
    

In [ ]:
for i in range(5):
    print(f'Epoch : {i+1} -----------------------------------------------------------------')
    train(train_dataloader, g_model, d_model, loss, optimizer_d, optimizer_g, 5)

Epoch : 1 -----------------------------------------------------------------
Descriminator Loss : 4.873494576209532e-09, Generator Loss : 6.316465905008284e-13
Epoch : 2 -----------------------------------------------------------------
Descriminator Loss : 7.058103089488583e-10, Generator Loss : 1.1959835519940315e-13
Epoch : 3 -----------------------------------------------------------------
Descriminator Loss : 1.467962364730748e-10, Generator Loss : 2.8898951509809603e-14
Epoch : 4 -----------------------------------------------------------------
